# SPFI ×4 — inference with every trained model, side by side (Colab GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SS1162/SR_amat_3/blob/riki/better-training/notebooks/03_all_models_inference_colab.ipynb)

Finds **every checkpoint** that `train_colab.ipynb` saved in the Drive folder (C8 / C16, recipe `v1` at 5000 and
20000 iterations, recipe `v2`, ...) and runs all of them on the **same images**:

1. **Scores on the validation split**: PSNR / SSIM of each model and of bicubic over the validation images
   (the 20% of EMPS `train.csv` the trainer held out; never trained on).
2. **Gallery**: for a few validation images, one figure with LR, bicubic, every model and HR, plus the same
   zoomed region of each.
3. **Your own images** (optional): upload microscope images and see what each model makes of them (no ground
   truth there, so pictures only).

**The test split is never downloaded.** Each image is degraded the way the models were trained (`degrade`:
Gaussian blur + bicubic ×4 downscale) and the models reconstruct it with the same 64×64 tiling as the app.

**Before you start:** Runtime → Change runtime type → **GPU**. Then Runtime → Run all.

## Settings
By default every `spfi_x4_final_*.pt` in the Drive folder is used (the *final* weights: each run trained for
exactly its iteration budget). `WHICH = "best"` uses the lowest-validation-loss weights instead.
`INCLUDE` limits the run to some files. If Drive does not mount, you are asked to upload the `.pt` files.

In [ ]:
import os
from pathlib import Path

# ---- Code: the v2 recipe and the memory-efficient attention live on this branch ----
CODE_REPO_URL = "https://github.com/SS1162/SR_amat_3.git"
CODE_BRANCH = "riki/better-training"

# ---- Data: public EMPS release (same as train_colab.ipynb); only train.csv images are extracted ----
EMPS_COMMIT = "dd8f86d714894fd08f3db80041893982ce058dd6"
EMPS_URL = f"https://codeload.github.com/by256/emps/zip/{EMPS_COMMIT}"
EMPS_SHA256 = "141462d290f9ac780bf9599ef5cb4b3e6708adfe6abafd1d4b9d6c786a6ad3f8"
SPLIT_SEED = 0            # must equal SPLIT_SEED used for training -> the same validation images

# ---- Models ----
CHECKPOINT_DIR = Path("/content/drive/MyDrive/spfi_checkpoints")
WHICH = "final"           # "final", "best" or "both"
INCLUDE = None            # e.g. ["spfi_x4_final_e8_s0_it20000.pt", "spfi_x4_final_e8_s0_v2_it10000.pt"]; None = all

# ---- Images ----
N_IMAGES = 6              # gallery size: validation images spread from hardest to easiest for bicubic
IMAGE_IDS = None          # or a list of validation image ids to show instead
SCORE_ALL_VAL = True      # PSNR / SSIM of every model over all validation images (a few minutes on a GPU)
UPLOAD_OWN = False        # True: also upload your own images and run every model on them

# ---- Inference (same tiling as the app / inference_colab.ipynb) ----
TILE, OVERLAP, BATCH_TILES = 64, 16, 16
SHAVE = 4                 # HR border pixels ignored by PSNR / SSIM
CROP = 128                # side of the zoomed region (HR pixels)

IN_COLAB = "google.colab" in __import__("sys").modules
WORK_DIR = Path("/content") if IN_COLAB else Path(os.environ.get("SR_WORK_DIR", Path.cwd() / "_all_models"))
CODE_DIR, DATA_DIR, OUTPUT_DIR = WORK_DIR / "SR_amat_3", WORK_DIR / "emps", WORK_DIR / "output"
# Local testing only: environment variables override the values above.
CODE_REPO_URL = os.environ.get("SR_REPO_URL") or CODE_REPO_URL
CODE_BRANCH = os.environ.get("SR_REF") or CODE_BRANCH
CHECKPOINT_DIR = Path(os.environ.get("SR_CKPT_DIR") or CHECKPOINT_DIR)
if os.environ.get("SR_N_IMAGES"):
    N_IMAGES = int(os.environ["SR_N_IMAGES"])
if os.environ.get("SR_MAX_VAL"):
    MAX_VAL = int(os.environ["SR_MAX_VAL"])
else:
    MAX_VAL = None
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Code, dependencies and data

In [ ]:
import subprocess
import sys

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pytorch_wavelets>=1.3", "PyWavelets>=1.6", "setuptools<81"], check=True)

In [ ]:
import hashlib
import time
import urllib.request
import zipfile


def git(*args, cwd=None, attempts=3):
    """Run git, retrying network hiccups."""
    for attempt in range(1, attempts + 1):
        result = subprocess.run(["git", *args], cwd=cwd, text=True, capture_output=True)
        if result.returncode == 0:
            return result.stdout.strip()
        if attempt < attempts:
            time.sleep(5 * attempt)
    raise RuntimeError(f"git {args[0]} failed:\n{result.stderr}")


def read_split(text):
    """Image ids (file stems) listed in train.csv / test.csv, one per line."""
    return [line.split(",")[0].strip() for line in text.splitlines() if line.strip()]


def extract_emps_train():
    """Download the pinned EMPS archive once and extract only the train.csv images into DATA_DIR/train/."""
    archive = DATA_DIR / "emps.zip"
    if not archive.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print("Downloading EMPS...")
        urllib.request.urlretrieve(EMPS_URL, archive)
    sha = hashlib.sha256(archive.read_bytes()).hexdigest()
    if sha != EMPS_SHA256:
        print(f"WARNING: archive SHA-256 {sha} differs from the recorded {EMPS_SHA256}")
    dest = DATA_DIR / "train"
    (dest / "images").mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        root = next(n for n in zf.namelist() if n.endswith("/train.csv"))[:-len("train.csv")]
        for name in ("train.csv", "test.csv"):
            (dest / name).write_bytes(zf.read(root + name))
        ids = read_split((dest / "train.csv").read_text())
        for image_id in ids:
            out = dest / "images" / f"{image_id}.png"
            if not out.exists():
                out.write_bytes(zf.read(f"{root}images/{image_id}.png"))
    return dest, ids


if not (CODE_DIR / ".git").exists():
    git("clone", "--quiet", "--no-checkout", CODE_REPO_URL, str(CODE_DIR))
git("fetch", "--quiet", "origin", CODE_BRANCH, cwd=CODE_DIR)
git("checkout", "--quiet", "--force", "--detach", "FETCH_HEAD", cwd=CODE_DIR)
print("code:", CODE_BRANCH, "@", git("log", "-1", "--format=%h %s", cwd=CODE_DIR))
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
for name in [m for m in sys.modules if m == "algo" or m.startswith("algo.")]:
    del sys.modules[name]   # re-import algo/ from the branch just checked out

from algo.train import split_files

EMPS_DIR, train_ids = extract_emps_train()
test_ids = set(read_split((EMPS_DIR / "test.csv").read_text()))
on_disk = sorted(p.stem for p in (EMPS_DIR / "images").glob("*.png"))
assert on_disk == sorted(train_ids), "images/ must hold exactly the train split"
assert not set(on_disk) & test_ids, "test images must never be used here"
_, val_ids = split_files(on_disk, SPLIT_SEED)
if MAX_VAL is not None:
    val_ids = val_ids[:MAX_VAL]
print(f"{len(val_ids)} validation images (never trained on); test split ({len(test_ids)} images) not downloaded")

## Find and load the models
Every checkpoint is loaded the way the local app loads it (`model_state` + `hparams`). The label is built from
the file name and the checkpoint itself: width, recipe (`v1` = batch 1 / constant learning rate, `v2` =
improved recipe) and the number of training iterations stored in the file.

In [ ]:
import re
import warnings

warnings.filterwarnings("ignore", message="pkg_resources is deprecated")

import pandas as pd
import torch

from algo.model import build_spfi

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU (slow)")

NAME = re.compile(r"spfi_x4_(final|best)_e(\d+)_s(\d+)(?:_(v\d+))?(?:_it(\d+))?\.pt")


def find_checkpoints():
    folder = CHECKPOINT_DIR
    if IN_COLAB and str(folder).startswith("/content/drive"):
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as e:   # mount failed: fall back to uploading the files
            print(f"Drive did not mount ({e}).")
            folder = None
    if folder is not None and folder.exists():
        found = sorted(p for p in folder.glob("spfi_x4_*.pt") if NAME.fullmatch(p.name))
        if found:
            return found
    if not IN_COLAB:
        return []
    from google.colab import files
    print("Upload the checkpoint files (.pt) — several at once is fine")
    up = WORK_DIR / "uploaded_checkpoints"
    up.mkdir(exist_ok=True)
    for name, data in files.upload().items():
        (up / name).write_bytes(data)
    return sorted(p for p in up.glob("*.pt") if NAME.fullmatch(p.name))


kinds = {"final": ("final",), "best": ("best",), "both": ("final", "best")}[WHICH]
paths = [p for p in find_checkpoints()
         if NAME.fullmatch(p.name)[1] in kinds and (INCLUDE is None or p.name in INCLUDE)]
assert paths, f"no checkpoint files found in {CHECKPOINT_DIR}"

MODELS, rows = {}, []
for p in paths:
    kind, dim, seed, recipe, _ = NAME.fullmatch(p.name).groups()
    recipe = recipe or "v1"
    ckpt = torch.load(p, map_location=device, weights_only=True)
    hp = ckpt["hparams"]
    model = build_spfi(scale=hp["scale"], embed_dim=hp["embed_dim"])
    model.load_state_dict(ckpt["model_state"])
    iters = ckpt.get("iteration") or 0
    label = f"C{hp['embed_dim']} {recipe} {iters / 1000:g}k" + (" best" if kind == "best" else "") + \
            (f" s{seed}" if seed != "0" else "")
    MODELS[label] = model.to(device).eval()
    rows.append({"model": label, "file": p.name, "width": hp["embed_dim"], "recipe": recipe,
                 "iterations": iters, "params": sum(q.numel() for q in model.parameters())})

order = sorted(rows, key=lambda r: (r["width"], r["recipe"], r["iterations"], r["model"]))
MODELS = {r["model"]: MODELS[r["model"]] for r in order}
MODEL_TABLE = pd.DataFrame(order).set_index("model")
SCALE = 4
display(MODEL_TABLE)
print("recipe v1 = batch 1, constant LR 1e-4; v2 = batch 16, warm-up + cosine LR 2e-4, EMA weights, flips/rotations")

## Scores on the validation split
PSNR / SSIM of each model on the whole image (a `SHAVE`-pixel border ignored), averaged over the validation
images, with bicubic as the reference. "beats bicubic" = on how many images the model's PSNR is higher.
These are full images, so the numbers are lower than the patch numbers printed during training.

In [ ]:
import numpy as np
import torch.nn.functional as F
from torchvision.transforms.functional import to_tensor

from algo.image_loading import load_image
from algo.preprocessing import normalize_image
from algo.train import degrade, psnr, ssim_loss


def tile_starts(n):
    if n <= TILE:
        return [0]
    return list(range(0, n - TILE, TILE - OVERLAP)) + [n - TILE]


@torch.no_grad()
def super_resolve(model, lr):
    # [1, 1, h, w] LR -> [1, 1, h*SCALE, w*SCALE], overlapping 64x64 tiles averaged (as in the app)
    _, _, h, w = lr.shape
    lr = F.pad(lr.to(device), (0, max(0, TILE - w), 0, max(0, TILE - h)), mode="replicate")
    H, W = lr.shape[-2:]
    coords = [(y, x) for y in tile_starts(H) for x in tile_starts(W)]
    out = torch.zeros(1, 1, H * SCALE, W * SCALE, device=device)
    hits = torch.zeros_like(out)
    t = TILE * SCALE
    for i in range(0, len(coords), BATCH_TILES):
        chunk = coords[i:i + BATCH_TILES]
        sr = model(torch.cat([lr[..., y:y + TILE, x:x + TILE] for y, x in chunk]))
        for (y, x), tile in zip(chunk, sr):
            out[..., y * SCALE:y * SCALE + t, x * SCALE:x * SCALE + t] += tile
            hits[..., y * SCALE:y * SCALE + t, x * SCALE:x * SCALE + t] += 1
    return (out / hits)[..., :h * SCALE, :w * SCALE].clamp(0, 1).cpu()


def bicubic(lr):
    return F.interpolate(lr, scale_factor=SCALE, mode="bicubic", align_corners=False).clamp(0, 1)


def load_val(image_id):
    # HR [1,1,H,W] cropped to a multiple of SCALE, and its LR made with the training degradation
    hr = to_tensor(normalize_image(load_image(EMPS_DIR / "images" / f"{image_id}.png")))
    _, H, W = hr.shape
    hr = hr[:, :H - H % SCALE, :W - W % SCALE]
    return hr.unsqueeze(0), degrade(hr, filename=image_id, scale=SCALE).unsqueeze(0)


shave = lambda t: t[..., SHAVE:-SHAVE, SHAVE:-SHAVE] if SHAVE else t
metrics = lambda out, hr: (psnr(shave(out), shave(hr)), 1 - ssim_loss(shave(out), shave(hr)).item())

score_ids = val_ids if SCORE_ALL_VAL else []
records = []
for n, image_id in enumerate(score_ids, 1):
    hr, lr = load_val(image_id)
    rec = {"image": image_id}
    rec["bicubic_psnr"], rec["bicubic_ssim"] = metrics(bicubic(lr), hr)
    for label, model in MODELS.items():
        rec[f"{label}_psnr"], rec[f"{label}_ssim"] = metrics(super_resolve(model, lr), hr)
    records.append(rec)
    if n % 10 == 0 or n == len(score_ids):
        print(f"{n}/{len(score_ids)} images")

if records:
    SCORES = pd.DataFrame(records).set_index("image")
    SCORES.to_csv(OUTPUT_DIR / "all_models_val_scores.csv")
    summary = []
    for label in ["bicubic", *MODELS]:
        p, s = SCORES[f"{label}_psnr"], SCORES[f"{label}_ssim"]
        row = {"method": label, "PSNR (dB)": p.mean(), "SSIM": s.mean(),
               "Δ PSNR vs bicubic": p.mean() - SCORES["bicubic_psnr"].mean()}
        if label != "bicubic":
            row["beats bicubic"] = f"{int((p > SCORES['bicubic_psnr']).sum())}/{len(p)}"
            row["params"] = MODEL_TABLE.loc[label, "params"]
        summary.append(row)
    SUMMARY = pd.DataFrame(summary).set_index("method")
    display(SUMMARY.round({"PSNR (dB)": 2, "SSIM": 4, "Δ PSNR vs bicubic": 2}))

    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(max(10, 1.6 * len(SUMMARY) * 2), 4))
    colors = ["#9AA0A6"] + ["#4C72B0" if MODEL_TABLE.loc[m, "width"] == 8 else "#DD8452" for m in MODELS]
    for ax, col in zip(axes, ("PSNR (dB)", "SSIM")):
        vals = SUMMARY[col]
        ax.bar(range(len(vals)), vals, color=colors)
        ax.set_xticks(range(len(vals)), vals.index, rotation=35, ha="right")
        lo, hi = vals.min(), vals.max()
        ax.set_ylim(lo - (hi - lo) * 0.6 - 1e-3, hi + (hi - lo) * 0.3 + 1e-3)
        for i, v in enumerate(vals):
            ax.text(i, v, f"{v:.2f}" if col.startswith("PSNR") else f"{v:.3f}", ha="center", va="bottom", fontsize=9)
        ax.set_title(f"{col} on {len(SCORES)} validation images (blue = C8, orange = C16)")
        ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "all_models_scores.png", dpi=120)
    plt.show()
else:
    SCORES = None
    print("SCORE_ALL_VAL = False: skipped")

## Gallery: every model on the same images
Top row: the whole image (LR enlarged ×4 without smoothing, bicubic, each model, HR) with the zoom box;
bottom row: the zoomed region. The box is placed automatically on the most detailed part of the image
(away from the burned-in labels at the edges). The full-size outputs are saved as PNG files in `gallery/`.

In [ ]:
import shutil

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

GALLERY_DIR = OUTPUT_DIR / "gallery"
GALLERY_DIR.mkdir(parents=True, exist_ok=True)
_SOBEL = torch.tensor([[1., 0., -1.], [2., 0., -2.], [1., 0., -1.]])


def grad_mag(img):
    g = torch.sqrt(F.conv2d(img, _SOBEL.view(1, 1, 3, 3), padding=1) ** 2
                   + F.conv2d(img, _SOBEL.t().reshape(1, 1, 3, 3), padding=1) ** 2 + 1e-12)
    return g[..., 1:-1, 1:-1]


def detail_crop(hr):
    # CROP x CROP window with the most edge detail, inside the central region (labels sit at the edges)
    g = grad_mag(hr)
    k = min(CROP, g.shape[-2], g.shape[-1])
    energy = F.avg_pool2d(g, k, stride=8)
    sat = F.avg_pool2d(((hr < 0.02) | (hr > 0.98)).float()[..., 1:-1, 1:-1], k, stride=8)
    energy = energy.masked_fill(sat > max(0.01, float(sat.min()) + 0.05), -1.0)
    H, W = hr.shape[-2:]
    ys = torch.arange(energy.shape[-2]) * 8 + 1
    xs = torch.arange(energy.shape[-1]) * 8 + 1
    my, mx = int(0.15 * H), int(0.15 * W)
    inside = ((ys >= my) & (ys + k <= H - my))[:, None] & ((xs >= mx) & (xs + k <= W - mx))[None, :]
    if inside.any():
        energy = energy.masked_fill(~inside, -2.0)
    iy, ix = divmod(int(energy.flatten().argmax()), energy.shape[-1])
    return min(iy * 8 + 1, H - k), min(ix * 8 + 1, W - k), k


def save_png(img, path):
    Image.fromarray((img[0, 0].clamp(0, 1).numpy() * 255).round().astype(np.uint8)).save(path)


def show(panels, crop, title, path):
    # panels: list of (name, image [1,1,H,W], caption)
    y, x, k = crop
    fig, axes = plt.subplots(2, len(panels), figsize=(2.9 * len(panels), 6.6))
    for col, (name, img, cap) in enumerate(panels):
        axes[0, col].imshow(img[0, 0], cmap="gray", vmin=0, vmax=1)
        axes[0, col].add_patch(Rectangle((x, y), k, k, fill=False, ec="#FFD400", lw=1.3))
        axes[0, col].set_title(name + ("\n" + cap if cap else ""), fontsize=9)
        axes[1, col].imshow(img[0, 0, y:y + k, x:x + k], cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        axes[1, col].set_title("zoom", fontsize=8)
    for ax in axes.flat:
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.savefig(path, dpi=100)
    plt.show()


if IMAGE_IDS:
    gallery = [i for i in IMAGE_IDS if i in val_ids]
    assert len(gallery) == len(IMAGE_IDS), "IMAGE_IDS must be validation images"
else:
    if SCORES is not None:
        bic = SCORES["bicubic_psnr"]
    else:
        bic = pd.Series({i: psnr(shave(bicubic(load_val(i)[1])), shave(load_val(i)[0])) for i in val_ids})
    ranked = bic.sort_values().index.tolist()           # hardest -> easiest for bicubic
    gallery = list(dict.fromkeys(ranked[round(q * (len(ranked) - 1))] for q in np.linspace(0, 1, N_IMAGES)))

for n, image_id in enumerate(gallery, 1):
    hr, lr = load_val(image_id)
    outs = {"bicubic": bicubic(lr), **{label: super_resolve(m, lr) for label, m in MODELS.items()}}
    cap = lambda img: "PSNR {:.2f}  SSIM {:.3f}".format(*metrics(img, hr))
    panels = ([("LR input (×4, no smoothing)", F.interpolate(lr, scale_factor=SCALE, mode="nearest"), "")]
              + [(name, img, cap(img)) for name, img in outs.items()] + [("HR (ground truth)", hr, "")])
    show(panels, detail_crop(hr), f"[{n}/{len(gallery)}] {image_id} — validation image (never trained on)",
         GALLERY_DIR / f"{n:02d}_{image_id}_all_models.png")
    for name, img in [("hr", hr), *outs.items()]:
        save_png(img, GALLERY_DIR / f"{n:02d}_{image_id}_{re.sub(r'[^A-Za-z0-9]+', '_', name)}.png")

## Your own images (optional, `UPLOAD_OWN = True`)
Upload grayscale microscope images (PNG / TIF). Each is treated as the **low-resolution input** and enlarged ×4
by bicubic and by every model. There is no ground truth, so there are no scores — only pictures.
Keep in mind the models learned one kind of degradation (Gaussian blur + bicubic downscale); a real
low-resolution image from a microscope can look different, and the models may handle it less well.
Large images take a while (a 1000×1000 input becomes 4000×4000).

In [ ]:
if UPLOAD_OWN and IN_COLAB:
    from google.colab import files

    own_dir = OUTPUT_DIR / "own_images"
    own_dir.mkdir(exist_ok=True)
    print("Upload one or more images")
    for name, data in files.upload().items():
        src = WORK_DIR / name
        src.write_bytes(data)
        lr = to_tensor(normalize_image(load_image(src))).unsqueeze(0)
        outs = {"bicubic": bicubic(lr), **{label: super_resolve(m, lr) for label, m in MODELS.items()}}
        h, w = outs["bicubic"].shape[-2:]
        k = min(CROP, h, w)
        crop = ((h - k) // 2, (w - k) // 2, k)    # centre of the image
        panels = ([("input (×4, no smoothing)", F.interpolate(lr, scale_factor=SCALE, mode="nearest"), "")]
                  + [(n_, img, "") for n_, img in outs.items()])
        stem = Path(name).stem
        show(panels, crop, f"{name} — your image, enlarged ×4 (no ground truth)", own_dir / f"{stem}_all_models.png")
        for n_, img in outs.items():
            save_png(img, own_dir / f"{stem}_{re.sub(r'[^A-Za-z0-9]+', '_', n_)}.png")
elif UPLOAD_OWN:
    print("UPLOAD_OWN works in Colab only")
else:
    print("UPLOAD_OWN = False: skipped")

## Download the results

In [ ]:
results_zip = shutil.make_archive(str(WORK_DIR / "all_models_results"), "zip", OUTPUT_DIR)
print("saved:", results_zip)
if IN_COLAB:
    from google.colab import files
    files.download(results_zip)

## Reading the results
* Compare models **of the same recipe and budget** for the width question (C8 vs C16), and the **same width**
  for the recipe question (v1 vs v2). Iterations are not comparable across recipes: one `v2` iteration uses
  16 patches, one `v1` iteration uses 1.
* PSNR favours smooth outputs; look at the zoomed crops for edges and particle boundaries.
* Validation images only, one training seed per model: a first look, not the final evaluation (the test
  split stays locked).